In [1]:
# Modules de base
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# Module relatif à Gurobi
from gurobipy import *

# Get and process files

## File processing functions

In [2]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import re
from typing import Any, Dict, List, Tuple, Optional, Union


# ----------------------------
# 1) File locator (decoupled)
# ----------------------------

def _format_density(density: Union[int, float, str]) -> str:
    """
    Convert density to the exact string used in folder/file names.
    Examples:
      1    -> "1"
      0.9  -> "0.9"
      0.95 -> "0.95"
      "4"  -> "4"
    """
    if isinstance(density, str):
        s = density.strip()
        # Normalize "1.0" -> "1" etc.
        try:
            f = float(s)
            if f.is_integer():
                return str(int(f))
            return s.rstrip("0").rstrip(".") if "." in s else s
        except ValueError:
            return s

    if isinstance(density, (int, float)):
        f = float(density)
        if f.is_integer():
            return str(int(f))
        # keep minimal representation (0.90 -> 0.9)
        s = repr(f)
        return s.rstrip("0").rstrip(".")
    return str(density)


def find_dat_file(
    base_dir: Union[str, Path],
    density: Union[int, float, str],
    p: int,
    h: int,
    test: Optional[int] = None,
    *,
    top_folders: Tuple[str, str] = ("Data", "Data with maintenance constraints"),
    folder_prefix: str = "d=",
    file_prefix: str = "DataCplex_",
) -> Path:
    """
    Search for the .dat matching density/p/h/(optional)test under:
      base_dir/Data/d=<density>/
      base_dir/Data with maintenance constraints/d=<density>/
    Returns the Path if uniquely found, otherwise raises an error.
    """
    base_dir = Path(base_dir)
    d_str = _format_density(density)

    candidates: List[Path] = []

    # Search both top folders (works even if one doesn't exist)
    for top in top_folders:
        d_folder = base_dir / top / f"{folder_prefix}{d_str}"
        if not d_folder.exists():
            continue

        if test is None:
            pattern = f"{file_prefix}density={d_str}_p={p}_h={h}_test_*.dat"
        else:
            pattern = f"{file_prefix}density={d_str}_p={p}_h={h}_test_{test}.dat"

        candidates.extend(sorted(d_folder.glob(pattern)))

    if not candidates:
        raise FileNotFoundError(
            f"No .dat file found for density={d_str}, p={p}, h={h}, test={test} under {base_dir}"
        )

    # If test=None we might match multiple; require uniqueness to be safe
    if len(candidates) > 1:
        msg = "\n".join(str(c) for c in candidates[:30])
        more = "" if len(candidates) <= 30 else f"\n... and {len(candidates)-30} more"
        raise FileExistsError(
            f"Multiple matching .dat files found for density={d_str}, p={p}, h={h}, test={test}:\n{msg}{more}\n"
            f"Pass test=<n> to select a unique file."
        )

    return candidates[0]


# ----------------------------
# 2) Parser (decoupled)
# ----------------------------

_DECL_RE = re.compile(r"(?ms)^\s*(\w+)\s*=\s*(.*?)\s*;\s*$")

def _parse_atom(token: str) -> Union[int, float, str]:
    token = token.strip()
    if token == "":
        return token

    # Try int then float
    try:
        if re.fullmatch(r"[+-]?\d+", token):
            return int(token)
        if re.fullmatch(r"[+-]?\d*\.\d+(?:[eE][+-]?\d+)?", token) or re.fullmatch(r"[+-]?\d+(?:[eE][+-]?\d+)", token):
            return float(token)
    except ValueError:
        pass

    # Otherwise string (e.g., airport codes A,B,C)
    return token


def _parse_set(value: str) -> List[Union[int, float, str]]:
    # value like "{A,B,C,}" or "{0,1,2,}"
    inner = value.strip()[1:-1].strip()
    if inner == "":
        return []
    parts = [p.strip() for p in inner.split(",")]
    parts = [p for p in parts if p != ""]
    return [_parse_atom(p) for p in parts]


def _parse_tuples_from_angle_brackets(value: str) -> List[Tuple[Union[int, float, str], ...]]:
    # Extract all <...> blocks, split by commas inside
    tuples: List[Tuple[Union[int, float, str], ...]] = []
    for body in re.findall(r"<([^<>]*)>", value, flags=re.S):
        fields = [f.strip() for f in body.split(",")]
        fields = [f for f in fields if f != ""]
        tuples.append(tuple(_parse_atom(f) for f in fields))
    return tuples


def _parse_matrix(value: str) -> List[List[Union[int, float]]]:
    """
    Parses something like:
      Cost =[
        [6804.0,6870.0,...,]
        [4536.0,4580.0,...,]
      ];
    Note: rows are NOT comma-separated in your file, so we scan bracket depth.
    """
    s = value.strip()
    if not (s.startswith("[") and s.endswith("]")):
        raise ValueError("Matrix must start with '[' and end with ']'")

    content = s[1:-1].strip()

    rows: List[str] = []
    i = 0
    n = len(content)
    while i < n:
        # Find next row '['
        while i < n and content[i].isspace():
            i += 1
        if i >= n:
            break
        if content[i] != "[":
            # skip unexpected chars
            i += 1
            continue

        # parse one row [...]
        depth = 0
        start = i
        while i < n:
            ch = content[i]
            if ch == "[":
                depth += 1
            elif ch == "]":
                depth -= 1
                if depth == 0:
                    end = i
                    rows.append(content[start + 1 : end])  # inside [...]
                    i += 1
                    break
            i += 1

    matrix: List[List[Union[int, float]]] = []
    for row in rows:
        nums = [x.strip() for x in row.replace("\n", " ").split(",")]
        nums = [x for x in nums if x != ""]
        parsed_row: List[Union[int, float]] = []
        for x in nums:
            atom = _parse_atom(x)
            if isinstance(atom, str):
                raise ValueError(f"Non-numeric value in matrix: {atom!r}")
            parsed_row.append(atom)
        matrix.append(parsed_row)

    return matrix


def parse_dat_file(dat_path: Union[str, Path]) -> Dict[str, Any]:
    """
    Parses an OPL/CPLEX-like .dat into a dict:
      - Sets {...} -> Python list
      - Scalars -> int/float/str
      - Tuple lists <...> -> list[tuple]
      - Matrices [...] with row brackets -> list[list[float/int]]
    """
    dat_path = Path(dat_path)
    text = dat_path.read_text(encoding="utf-8", errors="replace")

    out: Dict[str, Any] = {}

    # Split by semicolons safely by matching "name = ... ;" blocks (multiline)
    for m in _DECL_RE.finditer(text):
        name = m.group(1)
        raw = m.group(2).strip()

        if raw.startswith("{") and raw.endswith("}"):
            # could be a set, or a tuple-set like Flight = { <...> <...> }
            if "<" in raw and ">" in raw:
                out[name] = _parse_tuples_from_angle_brackets(raw)
            else:
                out[name] = _parse_set(raw)

        elif raw.startswith("[") and raw.endswith("]"):
            # could be matrix, or list of tuples inside brackets
            if "<" in raw and ">" in raw:
                out[name] = _parse_tuples_from_angle_brackets(raw)
            else:
                out[name] = _parse_matrix(raw)

        else:
            # scalar
            out[name] = _parse_atom(raw)

    return out



## Read a dataset file and extract data

In [20]:
base = Path.cwd()  # or Path(__file__).resolve().parent

# Find file by params:
#dat_file = find_dat_file(base, density=1, p=10, h=15, test=0)
dat_file = 'Data with maintenance constraints/d=4/DataCplex_density=1_p=10_h=15_test_0.dat'
print("Using file: ", dat_file)

data = parse_dat_file(dat_file)

print("Keys: ", data.keys())
print("NbFlights: ", data["Nbflight"])
print("First Flight tuple: ", data["Flight"][0])


Airports   = data["Airports"]
AirportMaintenance = data["Airportmaintenance"] # list of airports that offer maintenance
Nbflight   = data["Nbflight"]
Aircrafts  = data["Aircrafts"]
Flight     = data["Flight"]
Cost       = data["Cost"]
CostMaintenance = data["CostMaintenance"] # list of (list of maintenance costs for each maintenance airport) for each aircraft
CapacityMaintenance = data["capmaintenance"] # amount of aircrafts that can be maintained at once at one airport
Horizon = data["horizon"] # interval maximum sans maintenance
Aircraft   = data["Aircraft"]

print("NbAircrafts: ", len(Aircrafts))
print("NbAirports: ", len(Airports))
print("NbMaintenanceAirports: ", len(AirportMaintenance))
print("Maintenance horizon: ", Horizon, " days")
print("Maintenance capacity: ", CapacityMaintenance, " maintenances per day per maintenance airport")

print("")
print("Contents:")
print("Cost shape:", len(data["Cost"]), "x", len(data["Cost"][0]) if data["Cost"] else 0)
print("First Aircraft tuple:", data["Aircraft"][0])
print("Maintenance airports: ", AirportMaintenance)
print(CostMaintenance[0][0])
print(len(AirportMaintenance))
print("Maintenance capacity at one airport (max amount of aircrafts to be maintained at one time): ", CapacityMaintenance)

Using file:  Data with maintenance constraints/d=4/DataCplex_density=1_p=10_h=15_test_0.dat
Keys:  dict_keys(['Airports', 'Airportmaintenance', 'Nbflight', 'Aircrafts', 'Days', 'Flight', 'Cost', 'CostMaintenance', 'capmaintenance', 'horizon', 'Aircraft'])
NbFlights:  408
First Flight tuple:  (1, 'BCN', 'HAM', 525.0, 715.0, 1)
NbAircrafts:  10
NbAirports:  19
NbMaintenanceAirports:  11
Maintenance horizon:  8  days
Maintenance capacity:  4  maintenances per day per maintenance airport

Contents:
Cost shape: 408 x 10
First Aircraft tuple: (0, 'BCN')
Maintenance airports:  ['HAM', 'LGW', 'MAD', 'FRA', 'BCN', 'MUC', 'MXP', 'NCE', 'GLA', 'DUB', 'HEW']
6804.0
11
Maintenance capacity at one airport (max amount of aircrafts to be maintained at one time):  4


# Modeling

## Initialize maintenance parameters

In [4]:
maintenance_time = 100 # amount of time that one maintenance consumes

## Set up gurobi model

In [5]:
from gurobipy import Model, GRB, quicksum

turn_time = 0 

m = Model("tail_assignment_xij")

# Variables
x = m.addVars(range(Nbflight), range(len(Aircrafts)), vtype=GRB.BINARY, name="x")
maint = m.addVars(range(Nbflight), range(len(Aircrafts)), vtype=GRB.BINARY, name="m")

# Objective function
m.setObjective(
    # Cost of flights
    quicksum(Cost[i][a] * x[i, a] for i in range(Nbflight) for a in range(len(Aircrafts)))
    
    # Cost of maintenance
    + quicksum(
        maint[i, a]
        * CostMaintenance[a][AirportMaintenance.index(Flight[i][2])]
        for i in range(Nbflight)
        for a in range(len(Aircrafts))
        if Flight[i][2] in AirportMaintenance
    ),
    
    # We want to minimize the cost
    GRB.MINIMIZE
)

Set parameter Username
Set parameter LicenseID to value 2773683
Academic license - for non-commercial use only - expires 2027-02-02


### x Variable constraints

In [6]:
# Flight assignment constraint
# each flight assigned to exactly one aircraft
m.addConstrs(
    (quicksum(x[i, a] for a in range(len(Aircrafts))) == 1 for i in range(Nbflight)),
    name="assign"
)

# No overlapping flights per aircraft
# --- overlap constraints: if flights i and k overlap in time, they can't share the same aircraft ---
# Flight tuple: (id, origin, dest, dep, arr)  -> dep at [3], arr at [4]
for i in range(Nbflight):
    dep_i = float(Flight[i][3])
    arr_i = float(Flight[i][4])

    for k in range(i + 1, Nbflight):
        dep_k = float(Flight[k][3])
        arr_k = float(Flight[k][4])

        # intervals [dep_i, arr_i] and [dep_k, arr_k] overlap if:
        # dep_i < arr_k AND dep_k < arr_i
        overlap = (dep_i < arr_k) and (dep_k < arr_i)

        if overlap:
            for a in range(len(Aircrafts)):
                m.addConstr(
                    x[i, a] + x[k, a] <= 1,
                    name=f"overlap_{i}_{k}_{a}"
                )

# Airport inventory over time constraint
# time checkpoints
T = sorted({float(Flight[i][3]) for i in range(Nbflight)})

for a in range(len(Aircrafts)):
    init_ap = Aircraft[a][1]

    for p in Airports:
        init_here = 1 if p == init_ap else 0

        for tau in T:
            # Departures from p up to tau on aircraft a
            dep_up_to_tau = quicksum(
                x[i, a]
                for i in range(Nbflight)
                if Flight[i][1] == p and float(Flight[i][3]) <= tau
            )

            # Arrivals to p that are "available" by tau (arrival + turn_time <= tau)
            arr_up_to_tau = quicksum(
                x[i, a]
                for i in range(Nbflight)
                if Flight[i][2] == p and (float(Flight[i][4]) + turn_time) <= tau
            )

            # Airport inventory / balance over time:
            # cannot depart from p more times than (initially there) + (arrived there earlier)
            m.addConstr(
                dep_up_to_tau <= init_here + arr_up_to_tau,
                name=f"airport_balance_a{a}_p{p}_t{tau}"
            )


### maint Variable constraints

In [17]:
# Redundancy
for a in range(len(Aircraft)):
    for i in range(Nbflight):
        m.addConstr(
            maint[i, a] <= x[i, a],
            name = f"maint_redundancy_{i}_{a}"
        )

# Maintenance airports restriction
for a in range(len(Aircraft)):
    for i in range(Nbflight):
        arr_i = Flight[i][2]

        # No maintenance after a flight that does not arrive at a maintenance airport
        if not arr_i in AirportMaintenance:
            m.addConstr(
                maint[i, a] <= 0,
                name=f"maint_airport_{i}_{arr_i}"
            )
        
# Maintenance capacity restriction
# arrival_time[i] is a float (minutes)
arrival_times = [f[4] for f in Flight]
t_min = min(arrival_times)
t_max = max(arrival_times)

# one day time window
WINDOW = 24.0*60

# build time windows
time_windows = []
t = t_min
while t <= t_max:
    time_windows.append((t, t + WINDOW))
    t += WINDOW
print("Maximum capacity time windows: ", time_windows)

# add constraints
for ap in AirportMaintenance:
    for (t_start, t_end) in time_windows:
        m.addConstr(
            quicksum(
                maint[i, a]
                for a in range(len(Aircrafts))
                for i in range(len(Flight))
                if Flight[i][2] == ap
                and Flight[i][4] >= t_start
                and Flight[i][4] <  t_end
            ) <= CapacityMaintenance,
            name=f"MaintCap_{ap}_{t_start}"
        )

# Maximum interval between two maintenances
#
# Order flights by arrival time
arrival_times = list(set(arrival_times)) # remove duplicate arrival times
arrival_times.sort() # sort in ascending order
print("arrival times: ", arrival_times)

# For each arrival time add the constraints in the time windows after it
used_start_times = [] # avoid duplicate time windows
horizon_minutes = Horizon*24*60 # translate the horizon time to minutes

for arrival_time in [0]:# arrival_times:
    # Create time windows (size of maintenance interval)
    arrival_time_windows = []
    t = arrival_time
    while t <= t_max:
        if not t in used_start_times:
            arrival_time_windows.append((t, t + horizon_minutes))
            used_start_times.append(t)
        t += horizon_minutes

    # For each arrival time go through time windows
    for (t_start, t_end) in arrival_time_windows:
        print("Processing constraint for time window: ", t_start, " to ", t_end)
        # Require at least one maintenance for each aircraft
        m.addConstrs(
            (quicksum(
                maint[i, j]
                for i in range(Nbflight)
                if Flight[i][4] >= t_start
                and Flight[i][4] < t_end
            ) >= 1
            for j in range(len(Aircrafts))),
            name = f"MaxMaintInterval_{t_start}"
        )

Maximum capacity time windows:  [(710.0, 2150.0), (2150.0, 3590.0), (3590.0, 5030.0), (5030.0, 6470.0), (6470.0, 7910.0), (7910.0, 9350.0), (9350.0, 10790.0), (10790.0, 12230.0), (12230.0, 13670.0), (13670.0, 15110.0), (15110.0, 16550.0), (16550.0, 17990.0), (17990.0, 19430.0), (19430.0, 20870.0), (20870.0, 22310.0)]
arrival times:  [710.0, 715.0, 720.0, 745.0, 750.0, 815.0, 850.0, 910.0, 925.0, 955.0, 1015.0, 1105.0, 1110.0, 1115.0, 1155.0, 1255.0, 1330.0, 1350.0, 1400.0, 1410.0, 1415.0, 1435.0, 1505.0, 1525.0, 1550.0, 1605.0, 1635.0, 1705.0, 2150.0, 2155.0, 2180.0, 2190.0, 2265.0, 2280.0, 2295.0, 2340.0, 2350.0, 2365.0, 2385.0, 2455.0, 2490.0, 2555.0, 2565.0, 2595.0, 2640.0, 2675.0, 2760.0, 2775.0, 2785.0, 2855.0, 2860.0, 2870.0, 2940.0, 2980.0, 3050.0, 3060.0, 3065.0, 3530.0, 3590.0, 3595.0, 3600.0, 3730.0, 3780.0, 3790.0, 3805.0, 3890.0, 3895.0, 3910.0, 3920.0, 3930.0, 3995.0, 4035.0, 4110.0, 4185.0, 4200.0, 4230.0, 4380.0, 4395.0, 4420.0, 4530.0, 4610.0, 5035.0, 5040.0, 5070.0, 51

## Execute optimisation

In [21]:
from gurobipy import GRB

m.params.outputflag = 0

m.optimize()

if m.status == GRB.OPTIMAL:
    print("Optimal objective value:", m.objVal)
    
    for i in range(Nbflight):
        for j in range(len(Aircrafts)):
            if x[i, j].X > 0.5:   # variable selected
                print(f"Flight {Flight[i][0]} assigned to Aircraft {Aircraft[j][0]}")
else:
    print("No optimal solution, status =", m.status)

No optimal solution, status = 3


# Plot

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# --------- Build assigned flights (now keep origin/dest) ----------
assigned = {a: [] for a in range(len(Aircrafts))}
airports = set()

for i in range(Nbflight):
    for a in range(len(Aircrafts)):
        if x[i, a].X > 0.5:
            dep = float(Flight[i][3])
            arr = float(Flight[i][4])
            fid = Flight[i][0]
            origin = Flight[i][1]
            dest   = Flight[i][2]
            assigned[a].append((dep, arr, fid, origin, dest))
            airports.add(origin)
            airports.add(dest)

# sort flights per aircraft by departure time
for a in range(len(Aircrafts)):
    assigned[a].sort(key=lambda t: t[0])

# --------- Assign one color per airport ----------
airports = sorted(list(airports))
cmap = plt.get_cmap("tab20", len(airports))
airport_color = {ap: cmap(idx) for idx, ap in enumerate(airports)}

# --------- Plot ----------
row_spacing = 1.4   # increase spacing between aircraft rows
bar_height  = 0.7   # slightly taller bars

fig, ax = plt.subplots(figsize=(13, 0.45 * max(1, len(Aircrafts))))

y = 0
yticks = []
ylabels = []

for a in range(len(Aircrafts)):
    if not assigned[a]:
        continue

    yticks.append(y)
    ylabels.append(f"AC {Aircrafts[a]}")

    for (dep, arr, fid, origin, dest) in assigned[a]:
        duration = arr - dep
        mid = dep + duration / 2.0

        # left half = origin color
        ax.barh(
            y, mid - dep, left=dep, height=bar_height,
            color=airport_color[origin],
            edgecolor="black", linewidth=0.9
        )

        # right half = destination color
        ax.barh(
            y, arr - mid, left=mid, height=bar_height,
            color=airport_color[dest],
            edgecolor="black", linewidth=0.9
        )

        # flight ID ABOVE the block (centered)
        ax.text(
            dep + duration/2,
            y + bar_height/2 + 0.08,
            str(fid),
            ha="center",
            va="bottom",
            fontsize=7,
            fontweight="bold"
        )

    y += row_spacing

ax.set_yticks(yticks)
ax.set_yticklabels(ylabels)
ax.set_xlabel("Time")
ax.set_title("Assigned flights per aircraft (origin/destination colors)")

legend_elements = [
    Patch(facecolor=airport_color[ap], edgecolor='black', label=ap)
    for ap in airports
]

ax.legend(
    handles=legend_elements,
    title="Airports",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    borderaxespad=0
)

plt.tight_layout()
plt.show()